# Chapter 6.2: Data Parallelism & Expert Parallelism in SGLang

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Data Parallelism (DP)** in SGLang and the `DataParallelController`
2. **Master request routing** strategies across DP workers
3. **Learn Expert Parallelism (EP)** for Mixture-of-Experts (MoE) models
4. **Understand all-to-all communication** for expert dispatch
5. **Analyze load balancing** strategies across experts
6. **Explore hybrid parallelism** combining TP, DP, and EP
7. **Read source code** of SGLang's parallelism implementation
8. **Configure and profile** multi-GPU serving

## Prerequisites

- Understanding of tensor parallelism (Part 4/5)
- Familiarity with MoE model architecture
- Basic understanding of NCCL/distributed communication

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.2_parallelism.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.2_parallelism.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Parallelism Strategies Overview

When serving LLMs on multiple GPUs, there are several parallelism strategies:

```
+-------------------------------------------------------------------+
|                    Parallelism Strategies                          |
+-------------------------------------------------------------------+
|                                                                   |
|  Tensor Parallelism (TP)    Data Parallelism (DP)                |
|  - Split model layers       - Replicate full model               |
|  - Each GPU has a slice     - Each GPU handles different         |
|  - Requires all-reduce        requests independently            |
|  - Best for large models    - Best for throughput scaling        |
|                                                                   |
|  Expert Parallelism (EP)    Pipeline Parallelism (PP)            |
|  - Split experts across     - Split layers across GPUs          |
|    GPUs (for MoE models)    - Micro-batching for overlap        |
|  - Requires all-to-all      - Less communication overhead       |
|    communication                                                  |
+-------------------------------------------------------------------+
```

### Key Trade-offs

| Strategy | Communication | Memory Efficiency | Throughput Scaling |
|----------|--------------|-------------------|-------------------|
| TP | High (all-reduce per layer) | Good (model split) | Limited by comm |
| DP | Minimal (routing only) | Poor (model replicated) | Near-linear |
| EP | Medium (all-to-all) | Good (experts split) | Good for MoE |
| TP+DP | Moderate | Balanced | Best for large-scale |

## 2. Data Parallelism in SGLang

### 2.1 Architecture

SGLang's Data Parallelism replicates the entire model on each GPU group and routes requests to different replicas.

```
              Client Requests
                    |
                    v
        +------------------------+
        | DataParallelController |
        |                        |
        | - Request routing      |
        | - Load balancing       |
        | - Cache-aware dispatch |
        +------------------------+
           /        |        \
          v         v         v
    +---------+ +---------+ +---------+
    | Worker 0| | Worker 1| | Worker 2|
    | (GPU 0) | | (GPU 1) | | (GPU 2) |
    | Full    | | Full    | | Full    |
    | Model   | | Model   | | Model   |
    | Copy    | | Copy    | | Copy    |
    +---------+ +---------+ +---------+
```

### 2.2 When to Use DP vs TP

- **DP**: Model fits on a single GPU. Want to scale throughput with more GPUs.
- **TP**: Model too large for a single GPU. Must split across GPUs.
- **TP+DP**: Model needs TP across N GPUs, have M*N GPUs total. Use TP within groups, DP across groups.

### 2.3 Source Code: DataParallelController

The `DataParallelController` is defined in `python/sglang/srt/managers/data_parallel_controller.py`.

```python
# From sglang/srt/managers/data_parallel_controller.py

class DataParallelController:
    """
    Manages multiple DP worker groups.
    Each worker group can itself use TP internally.
    """
    
    def __init__(self, server_args, port_args):
        self.dp_size = server_args.dp_size  # Number of DP replicas
        self.tp_size = server_args.tp_size  # TP degree within each replica
        
        # Launch worker processes
        self.workers = []
        for dp_rank in range(self.dp_size):
            # Each DP worker is a complete scheduler+model group
            gpu_ids = self._get_gpu_ids(dp_rank)
            worker = self._launch_worker(dp_rank, gpu_ids)
            self.workers.append(worker)
        
        # Communication channels (ZMQ sockets)
        self.dispatch_lookup = {}  # req_id -> worker_idx
    
    def _get_gpu_ids(self, dp_rank):
        """
        Assign GPU IDs for a DP rank.
        E.g., with dp_size=2, tp_size=2:
          dp_rank 0 -> GPUs [0, 1]
          dp_rank 1 -> GPUs [2, 3]
        """
        start = dp_rank * self.tp_size
        return list(range(start, start + self.tp_size))
    
    def dispatch_request(self, request):
        """
        Route a request to the best DP worker.
        Uses round-robin or cache-aware routing.
        """
        if self.dispatch_policy == "round_robin":
            worker_idx = self.next_worker
            self.next_worker = (self.next_worker + 1) % self.dp_size
        elif self.dispatch_policy == "cache_aware":
            worker_idx = self._find_best_cache_match(request)
        
        self.dispatch_lookup[request.rid] = worker_idx
        self.workers[worker_idx].send(request)
```

**Key design decisions:**
1. Each DP worker is a completely independent scheduler (no shared state)
2. Communication is via ZMQ sockets (lightweight IPC)
3. Cache-aware routing inspects radix tree state for optimal dispatch

In [ ]:
# Simulation: Data Parallel Controller
import random
from collections import defaultdict
from typing import List, Dict

class SimulatedDPWorker:
    """A simulated DP worker with its own cache and processing queue."""
    
    def __init__(self, worker_id, tp_size=1):
        self.worker_id = worker_id
        self.tp_size = tp_size
        self.cache = {}  # prefix_hash -> True (simplified cache)
        self.queue = []
        self.processed = 0
        self.cache_hits = 0
        self.total_compute_time = 0  # In arbitrary units
    
    def has_prefix(self, prefix_hash):
        return prefix_hash in self.cache
    
    def process_request(self, request):
        prefix_hash = hash(tuple(request["prefix"]))
        
        if prefix_hash in self.cache:
            # Cache hit: only compute new tokens
            compute_time = request["new_tokens"]
            self.cache_hits += 1
        else:
            # Cache miss: compute full prefill
            compute_time = request["prefix_len"] + request["new_tokens"]
            self.cache[prefix_hash] = True
        
        self.processed += 1
        self.total_compute_time += compute_time
        return compute_time


class SimulatedDPController:
    """Simulated DataParallelController."""
    
    def __init__(self, n_workers, tp_size=1):
        self.workers = [SimulatedDPWorker(i, tp_size) for i in range(n_workers)]
        self.n_workers = n_workers
        self.rr_counter = 0
    
    def dispatch(self, request, policy="round_robin"):
        if policy == "round_robin":
            worker_idx = self.rr_counter % self.n_workers
            self.rr_counter += 1
        elif policy == "cache_aware":
            prefix_hash = hash(tuple(request["prefix"]))
            # Find worker with cache hit, or least loaded
            for i, w in enumerate(self.workers):
                if w.has_prefix(prefix_hash):
                    worker_idx = i
                    break
            else:
                # No cache hit: route to least loaded worker
                worker_idx = min(range(self.n_workers), 
                                key=lambda i: self.workers[i].processed)
        elif policy == "hash":
            prefix_hash = hash(tuple(request["prefix"]))
            worker_idx = prefix_hash % self.n_workers
        
        return self.workers[worker_idx].process_request(request)
    
    def get_stats(self):
        total_hits = sum(w.cache_hits for w in self.workers)
        total_processed = sum(w.processed for w in self.workers)
        total_compute = sum(w.total_compute_time for w in self.workers)
        
        return {
            "total_processed": total_processed,
            "cache_hit_rate": total_hits / total_processed if total_processed > 0 else 0,
            "total_compute": total_compute,
            "per_worker": [
                {"id": w.worker_id, "processed": w.processed, 
                 "hits": w.cache_hits, "compute": w.total_compute_time}
                for w in self.workers
            ]
        }

print("SimulatedDPController defined.")

In [ ]:
# Generate workload and compare routing policies

def generate_dp_workload(n_requests=500, n_system_prompts=5, n_users=50):
    """Generate requests with shared system prompts."""
    system_prompts = [f"sys_prompt_{i}" for i in range(n_system_prompts)]
    requests = []
    
    for _ in range(n_requests):
        # Zipf-like distribution for system prompts
        weights = [1.0 / (i + 1) for i in range(n_system_prompts)]
        sys_prompt = random.choices(system_prompts, weights=weights)[0]
        user = f"user_{random.randint(0, n_users-1)}"
        
        requests.append({
            "prefix": [sys_prompt, user],
            "prefix_len": random.randint(100, 500),
            "new_tokens": random.randint(10, 100),
        })
    
    return requests

random.seed(42)
workload = generate_dp_workload(500)

print("=== Data Parallel Routing Policy Comparison ===")
print(f"Workers: 4, Requests: {len(workload)}\n")

for policy in ["round_robin", "hash", "cache_aware"]:
    controller = SimulatedDPController(n_workers=4)
    total_compute = 0
    for req in workload:
        total_compute += controller.dispatch(req, policy=policy)
    
    stats = controller.get_stats()
    print(f"\n--- {policy.upper()} ---")
    print(f"  Cache hit rate: {stats['cache_hit_rate']:.1%}")
    print(f"  Total compute units: {stats['total_compute']}")
    print(f"  Per-worker load distribution:")
    for w in stats["per_worker"]:
        print(f"    Worker {w['id']}: {w['processed']} reqs, {w['hits']} hits, {w['compute']} compute")

## 3. Expert Parallelism for MoE Models

### 3.1 Mixture-of-Experts Architecture

MoE (Mixture-of-Experts) models replace dense FFN layers with a set of expert networks and a gating mechanism:

```
           Input tokens
                |
                v
        +---------------+
        |  Gate/Router  |  --> Selects top-K experts per token
        +---------------+
         /    |     \    \
        v     v      v    v
    +-----+ +-----+ +-----+ +-----+
    |Exp 0| |Exp 1| |Exp 2| |Exp 3|  ... (e.g., 64 experts)
    +-----+ +-----+ +-----+ +-----+
        \     |      /    /
         v    v     v    v
        +----------------+
        | Weighted Sum   |  --> Combine expert outputs
        +----------------+
                |
                v
           Output tokens
```

**Popular MoE models:**
- Mixtral 8x7B (8 experts, top-2 routing)
- DeepSeek-V2/V3 (160 experts, top-6 routing)
- Qwen MoE

### 3.2 Why Expert Parallelism?

With 64+ experts, each expert is relatively small, but the total parameter count is large. Expert Parallelism distributes experts across GPUs:

```
64 experts distributed across 8 GPUs:

GPU 0: Experts [0-7]     GPU 4: Experts [32-39]
GPU 1: Experts [8-15]    GPU 5: Experts [40-47]
GPU 2: Experts [16-23]   GPU 6: Experts [48-55]
GPU 3: Experts [24-31]   GPU 7: Experts [56-63]
```

### 3.3 All-to-All Communication for Expert Dispatch

The critical operation in EP is **all-to-all communication**: each GPU must send tokens to the GPU hosting the selected expert, and receive results back.

```
Step 1: Gate computes expert assignments
  Token 0 -> Expert 3 (GPU 0)      Token 4 -> Expert 15 (GPU 1)
  Token 1 -> Expert 45 (GPU 5)     Token 5 -> Expert 3 (GPU 0)
  Token 2 -> Expert 12 (GPU 1)     Token 6 -> Expert 60 (GPU 7)
  Token 3 -> Expert 3 (GPU 0)      Token 7 -> Expert 45 (GPU 5)

Step 2: All-to-All dispatch (send tokens to correct GPU)
  GPU 0 receives: tokens [0, 3, 5] for Expert 3
  GPU 1 receives: tokens [2, 4] for Experts 12, 15
  GPU 5 receives: tokens [1, 7] for Expert 45
  GPU 7 receives: token [6] for Expert 60

Step 3: Each GPU processes its local experts

Step 4: All-to-All combine (send results back)
```

### 3.4 All-to-All vs All-Reduce

| Operation | Pattern | Used In | Communication Volume |
|-----------|---------|---------|---------------------|
| All-Reduce | All GPUs contribute, all receive sum | TP | O(N * data_size) |
| All-to-All | Each GPU sends different data to each GPU | EP | O(N * data_size / N) per GPU |

All-to-All is more communication-efficient for EP because each token only goes to its assigned GPU(s).

In [ ]:
# Simulate Expert Parallelism with All-to-All Communication
import numpy as np

class MoESimulator:
    """Simulate Mixture-of-Experts with Expert Parallelism."""
    
    def __init__(self, n_experts, n_gpus, top_k=2):
        self.n_experts = n_experts
        self.n_gpus = n_gpus
        self.top_k = top_k
        self.experts_per_gpu = n_experts // n_gpus
        
        # Map expert -> GPU
        self.expert_to_gpu = {}
        for i in range(n_experts):
            self.expert_to_gpu[i] = i // self.experts_per_gpu
    
    def gate(self, n_tokens, balance_factor=0.0):
        """
        Simulate gating: assign top-K experts to each token.
        balance_factor: 0.0 = natural routing, 1.0 = perfectly balanced
        """
        assignments = []
        for t in range(n_tokens):
            # Generate logits (some experts are more popular)
            logits = np.random.randn(self.n_experts)
            
            # Add popularity bias (some experts get more traffic)
            # This simulates real-world imbalanced routing
            popularity = np.array([
                2.0 if i < self.n_experts // 8 else  # Top 12.5% experts are 2x popular
                1.0 if i < self.n_experts // 2 else  # Next 37.5% are normal
                0.5                                   # Bottom 50% are less popular
                for i in range(self.n_experts)
            ])
            
            # Blend natural vs balanced
            if balance_factor > 0:
                balanced = np.ones(self.n_experts)  # Equal probability
                logits = (1 - balance_factor) * logits * popularity + balance_factor * balanced
            else:
                logits = logits * popularity
            
            # Select top-K experts
            top_k_experts = np.argsort(logits)[-self.top_k:]
            assignments.append(list(top_k_experts))
        
        return assignments
    
    def analyze_dispatch(self, assignments):
        """Analyze the all-to-all dispatch pattern."""
        gpu_load = defaultdict(int)  # GPU -> number of token-expert pairs
        expert_load = defaultdict(int)
        
        for token_experts in assignments:
            for expert_id in token_experts:
                gpu_id = self.expert_to_gpu[expert_id]
                gpu_load[gpu_id] += 1
                expert_load[expert_id] += 1
        
        loads = [gpu_load.get(i, 0) for i in range(self.n_gpus)]
        max_load = max(loads)
        min_load = min(loads)
        avg_load = sum(loads) / len(loads)
        
        return {
            "gpu_loads": loads,
            "max_load": max_load,
            "min_load": min_load,
            "avg_load": avg_load,
            "load_imbalance": max_load / avg_load if avg_load > 0 else 0,
            "expert_loads": dict(expert_load)
        }

# Run simulation
np.random.seed(42)
moe = MoESimulator(n_experts=64, n_gpus=8, top_k=2)

print("=== Expert Parallelism Simulation ===")
print(f"Experts: {moe.n_experts}, GPUs: {moe.n_gpus}, Top-K: {moe.top_k}")
print(f"Experts per GPU: {moe.experts_per_gpu}\n")

# Without load balancing
assignments = moe.gate(n_tokens=1000, balance_factor=0.0)
stats = moe.analyze_dispatch(assignments)

print("--- Without Load Balancing ---")
print(f"GPU loads: {stats['gpu_loads']}")
print(f"Load imbalance ratio: {stats['load_imbalance']:.2f}x (1.0 = perfect)")
print(f"Max GPU load: {stats['max_load']}, Min: {stats['min_load']}")

# With load balancing
assignments_balanced = moe.gate(n_tokens=1000, balance_factor=0.5)
stats_balanced = moe.analyze_dispatch(assignments_balanced)

print("\n--- With Load Balancing (factor=0.5) ---")
print(f"GPU loads: {stats_balanced['gpu_loads']}")
print(f"Load imbalance ratio: {stats_balanced['load_imbalance']:.2f}x")
print(f"Max GPU load: {stats_balanced['max_load']}, Min: {stats_balanced['min_load']}")

In [ ]:
# Visualize GPU load distribution
try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

if HAS_PLT:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # GPU load comparison
    x = range(moe.n_gpus)
    width = 0.35
    ax1.bar([i - width/2 for i in x], stats['gpu_loads'], width, label='Unbalanced', color='#e74c3c')
    ax1.bar([i + width/2 for i in x], stats_balanced['gpu_loads'], width, label='Balanced', color='#2ecc71')
    ax1.axhline(y=stats['avg_load'], color='gray', linestyle='--', alpha=0.5, label='Avg')
    ax1.set_xlabel('GPU ID')
    ax1.set_ylabel('Token-Expert Pairs')
    ax1.set_title('GPU Load Distribution')
    ax1.legend()
    
    # Expert popularity distribution
    expert_loads = sorted(stats['expert_loads'].items())
    expert_ids = [e[0] for e in expert_loads]
    expert_counts = [e[1] for e in expert_loads]
    ax2.bar(expert_ids, expert_counts, color='#3498db', alpha=0.7)
    ax2.set_xlabel('Expert ID')
    ax2.set_ylabel('Tokens Routed')
    ax2.set_title('Expert Load Distribution (Unbalanced)')
    
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available.")

### 3.5 Load Balancing Strategies for MoE

Load imbalance is the primary performance bottleneck in EP. Several strategies address this:

**1. Auxiliary Load Balancing Loss (Training Time)**
```python
# Added during training to encourage uniform expert usage
def load_balance_loss(gate_logits, expert_assignments):
    # f_i = fraction of tokens assigned to expert i
    f = expert_assignments.float().mean(dim=0)
    # p_i = average gate probability for expert i
    p = gate_logits.softmax(dim=-1).mean(dim=0)
    return (f * p).sum() * n_experts
```

**2. Token Dropping (Inference Time)**
```python
# Drop tokens from overloaded experts (capacity factor)
expert_capacity = int(n_tokens * top_k / n_experts * capacity_factor)
# Typically capacity_factor = 1.25 to allow 25% overflow
```

**3. Expert Choice Routing (Inference Time)**
```python
# Instead of tokens choosing experts, experts choose tokens
# This guarantees balanced load
for expert_id in range(n_experts):
    scores = gate_logits[:, expert_id]
    top_tokens = scores.topk(capacity)  # Expert picks its top tokens
```

In [ ]:
# Simulate Token Dropping and Expert Choice routing

class AdvancedMoERouter:
    """Advanced MoE routing with load balancing."""
    
    def __init__(self, n_experts, top_k, capacity_factor=1.25):
        self.n_experts = n_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor
    
    def standard_routing(self, gate_logits):
        """Standard top-K routing (can be imbalanced)."""
        n_tokens = gate_logits.shape[0]
        assignments = []
        expert_counts = defaultdict(int)
        
        for t in range(n_tokens):
            top_k = np.argsort(gate_logits[t])[-self.top_k:]
            assignments.append(list(top_k))
            for e in top_k:
                expert_counts[e] += 1
        
        return assignments, expert_counts
    
    def token_dropping_routing(self, gate_logits):
        """Top-K routing with capacity-limited token dropping."""
        n_tokens = gate_logits.shape[0]
        capacity = int(n_tokens * self.top_k / self.n_experts * self.capacity_factor)
        
        assignments = []
        expert_counts = defaultdict(int)
        dropped_tokens = 0
        
        for t in range(n_tokens):
            top_k = np.argsort(gate_logits[t])[-self.top_k:]
            assigned = []
            for e in top_k:
                if expert_counts[e] < capacity:
                    assigned.append(e)
                    expert_counts[e] += 1
                else:
                    dropped_tokens += 1
            assignments.append(assigned)
        
        return assignments, expert_counts, dropped_tokens
    
    def expert_choice_routing(self, gate_logits):
        """Expert choice: each expert picks its top tokens."""
        n_tokens = gate_logits.shape[0]
        capacity = int(n_tokens * self.top_k / self.n_experts)
        
        assignments = [[] for _ in range(n_tokens)]
        expert_counts = defaultdict(int)
        
        for e in range(self.n_experts):
            scores = gate_logits[:, e]
            top_tokens = np.argsort(scores)[-capacity:]
            for t in top_tokens:
                assignments[t].append(e)
                expert_counts[e] += 1
        
        return assignments, expert_counts

# Generate gate logits with popularity bias
np.random.seed(42)
n_tokens = 512
n_experts = 32

# Create biased logits (some experts preferred)
base_logits = np.random.randn(n_tokens, n_experts)
popularity_bias = np.array([2.0 if i < 4 else 0.5 if i > 24 else 1.0 for i in range(n_experts)])
gate_logits = base_logits * popularity_bias

router = AdvancedMoERouter(n_experts=n_experts, top_k=2)

# Standard routing
std_assign, std_counts = router.standard_routing(gate_logits)
std_loads = [std_counts.get(i, 0) for i in range(n_experts)]

# Token dropping
drop_assign, drop_counts, dropped = router.token_dropping_routing(gate_logits)
drop_loads = [drop_counts.get(i, 0) for i in range(n_experts)]

# Expert choice
ec_assign, ec_counts = router.expert_choice_routing(gate_logits)
ec_loads = [ec_counts.get(i, 0) for i in range(n_experts)]

print("=== MoE Routing Strategy Comparison ===")
print(f"Tokens: {n_tokens}, Experts: {n_experts}, Top-K: 2\n")

print(f"Standard:       max={max(std_loads)}, min={min(std_loads)}, "
      f"imbalance={max(std_loads)/np.mean(std_loads):.2f}x")
print(f"Token Dropping: max={max(drop_loads)}, min={min(drop_loads)}, "
      f"imbalance={max(drop_loads)/np.mean(drop_loads):.2f}x, "
      f"dropped={dropped} tokens")
print(f"Expert Choice:  max={max(ec_loads)}, min={min(ec_loads)}, "
      f"imbalance={max(ec_loads)/np.mean(ec_loads):.2f}x")

### 3.6 Source Code: Expert Parallelism in SGLang

SGLang implements EP for MoE models through integration with specialized kernels.

```python
# From sglang/srt/layers/moe/fused_moe_triton/fused_moe.py

def fused_experts(
    hidden_states: torch.Tensor,   # [num_tokens, hidden_dim]
    w1: torch.Tensor,              # [num_experts, inter_dim, hidden_dim]
    w2: torch.Tensor,              # [num_experts, hidden_dim, inter_dim]
    topk_weights: torch.Tensor,    # [num_tokens, top_k]
    topk_ids: torch.Tensor,        # [num_tokens, top_k]
    ...
):
    """
    Fused MoE expert computation.
    
    For EP:
    1. Dispatch tokens to correct GPU (all-to-all)
    2. Run local experts on each GPU
    3. Combine results (all-to-all)
    """
    # Sort tokens by expert assignment for coalesced memory access
    sorted_token_ids, expert_ids, num_tokens_per_expert = (
        moe_align_block_size(topk_ids, block_size, num_experts)
    )
    
    # Launch fused Triton kernel
    invoke_fused_moe_kernel(
        hidden_states, w1, w2,
        sorted_token_ids, expert_ids,
        topk_weights, ...
    )
```

**Key insight:** The `moe_align_block_size` function sorts tokens by expert assignment. This ensures that tokens for the same expert are contiguous in memory, enabling efficient batched matrix multiplication.

In [ ]:
# Simulate the token sorting and expert dispatch

def simulate_moe_dispatch(n_tokens, n_experts, top_k, n_gpus):
    """Simulate the full MoE dispatch pipeline for EP."""
    experts_per_gpu = n_experts // n_gpus
    
    # Step 1: Gate computation
    gate_logits = np.random.randn(n_tokens, n_experts)
    topk_ids = np.argsort(gate_logits, axis=1)[:, -top_k:]  # [n_tokens, top_k]
    topk_weights = np.take_along_axis(
        np.exp(gate_logits) / np.exp(gate_logits).sum(axis=1, keepdims=True),
        topk_ids, axis=1
    )  # Softmax weights for top-K
    
    # Step 2: Compute dispatch table (which GPU gets which tokens)
    dispatch_table = defaultdict(list)  # gpu_id -> [(token_id, expert_id, weight)]
    
    for t in range(n_tokens):
        for k in range(top_k):
            expert_id = topk_ids[t, k]
            gpu_id = expert_id // experts_per_gpu
            dispatch_table[gpu_id].append((t, expert_id, topk_weights[t, k]))
    
    # Step 3: Communication volume analysis
    total_pairs = n_tokens * top_k
    print(f"\n--- Dispatch Analysis ---")
    print(f"Total token-expert pairs: {total_pairs}")
    print(f"\nPer-GPU dispatch:")
    
    max_pairs = 0
    for gpu in range(n_gpus):
        pairs = len(dispatch_table[gpu])
        max_pairs = max(max_pairs, pairs)
        local_experts = set(p[1] for p in dispatch_table[gpu])
        print(f"  GPU {gpu}: {pairs:4d} pairs, {len(local_experts)} active experts")
    
    # Step 4: Compute communication volume
    hidden_dim = 4096  # Typical hidden dimension
    bytes_per_token = hidden_dim * 2  # FP16
    
    comm_volume = total_pairs * bytes_per_token * 2  # dispatch + combine
    print(f"\nCommunication volume: {comm_volume / 1e6:.1f} MB")
    print(f"Per-GPU max load: {max_pairs} pairs")
    print(f"Ideal load: {total_pairs / n_gpus:.0f} pairs/GPU")
    print(f"Load imbalance: {max_pairs / (total_pairs / n_gpus):.2f}x")

np.random.seed(42)
print("=== MoE Dispatch Simulation ===")

# Mixtral-like configuration
print("\n[Mixtral-like: 8 experts, 2 GPUs, top-2]")
simulate_moe_dispatch(n_tokens=256, n_experts=8, top_k=2, n_gpus=2)

# DeepSeek-like configuration  
print("\n\n[DeepSeek-like: 64 experts, 8 GPUs, top-6]")
simulate_moe_dispatch(n_tokens=256, n_experts=64, top_k=6, n_gpus=8)

## 4. Hybrid Parallelism: TP + DP + EP

### 4.1 Why Hybrid?

Real-world deployments often combine multiple parallelism strategies:

```
Example: DeepSeek-V3 on 64 GPUs

Total GPUs: 64
  TP = 8 (model split across 8 GPUs within a node)
  EP = 8 (experts distributed across 8 nodes)  
  DP = 1 (or more, if you have even more GPUs)

Within a TP group (8 GPUs, 1 node):
  - All-reduce for attention + dense layers
  - Each GPU has 1/8 of attention heads

Across EP groups (8 nodes):
  - All-to-all for MoE expert dispatch
  - Each node hosts 8 experts

Across DP groups:
  - Independent request processing
  - No communication during inference
```

### 4.2 Communication Hierarchy

```
                    DP Group 0                    DP Group 1
              +-------------------+         +-------------------+
    EP Node 0 | TP: GPU 0-7      |         | TP: GPU 32-39    |
              | All-reduce (NVLink)|        | All-reduce        |
              +-------------------+         +-------------------+
                    |   All-to-All (InfiniBand)      |
    EP Node 1 | TP: GPU 8-15     |         | TP: GPU 40-47    |
              +-------------------+         +-------------------+
                    |
    EP Node 2 | TP: GPU 16-23    |
              +-------------------+
                    |
    EP Node 3 | TP: GPU 24-31    |
              +-------------------+
```

**Communication costs:**
- TP all-reduce: fast (NVLink within node, ~600 GB/s)
- EP all-to-all: slower (InfiniBand across nodes, ~200 GB/s)
- DP routing: negligible (just request dispatch)

In [ ]:
# Hybrid Parallelism Configuration Calculator

class HybridParallelismConfig:
    """Calculate and validate hybrid parallelism configurations."""
    
    def __init__(self, total_gpus, tp_size, ep_size=1, dp_size=1):
        assert total_gpus == tp_size * ep_size * dp_size, \
            f"GPUs don't match: {total_gpus} != {tp_size} * {ep_size} * {dp_size}"
        
        self.total_gpus = total_gpus
        self.tp_size = tp_size
        self.ep_size = ep_size
        self.dp_size = dp_size
    
    def get_gpu_assignment(self):
        """Show which GPU belongs to which group."""
        assignments = []
        gpu_id = 0
        for dp in range(self.dp_size):
            for ep in range(self.ep_size):
                tp_gpus = list(range(gpu_id, gpu_id + self.tp_size))
                assignments.append({
                    "dp_rank": dp,
                    "ep_rank": ep,
                    "tp_gpus": tp_gpus
                })
                gpu_id += self.tp_size
        return assignments
    
    def estimate_communication(self, batch_size, seq_len, hidden_dim, n_layers, n_experts):
        """Estimate communication volume per forward pass."""
        bytes_per_element = 2  # FP16
        
        # TP all-reduce per layer: 2 * batch * seq * hidden * bytes (for reduce-scatter + all-gather)
        tp_comm_per_layer = 2 * batch_size * seq_len * hidden_dim * bytes_per_element
        tp_comm_total = tp_comm_per_layer * n_layers
        
        # EP all-to-all per MoE layer: 2 * batch * seq * hidden * bytes (dispatch + combine)
        n_moe_layers = n_layers  # Assuming every layer is MoE
        ep_comm_per_layer = 2 * batch_size * seq_len * hidden_dim * bytes_per_element
        ep_comm_total = ep_comm_per_layer * n_moe_layers
        
        return {
            "tp_comm_GB": tp_comm_total / 1e9,
            "ep_comm_GB": ep_comm_total / 1e9,
            "total_comm_GB": (tp_comm_total + ep_comm_total) / 1e9
        }
    
    def print_config(self):
        print(f"=== Hybrid Parallelism Configuration ===")
        print(f"Total GPUs: {self.total_gpus}")
        print(f"  TP size: {self.tp_size} (model sharding within group)")
        print(f"  EP size: {self.ep_size} (expert distribution across groups)")
        print(f"  DP size: {self.dp_size} (independent replicas)")
        print(f"\nGPU Assignments:")
        for a in self.get_gpu_assignment():
            print(f"  DP={a['dp_rank']}, EP={a['ep_rank']}: GPUs {a['tp_gpus']}")

# Example configurations
print("Configuration 1: 8 GPUs, TP=8 (single large model)")
config1 = HybridParallelismConfig(total_gpus=8, tp_size=8, ep_size=1, dp_size=1)
config1.print_config()

print("\n" + "="*60)
print("\nConfiguration 2: 8 GPUs, TP=2, DP=4 (throughput scaling)")
config2 = HybridParallelismConfig(total_gpus=8, tp_size=2, ep_size=1, dp_size=4)
config2.print_config()

print("\n" + "="*60)
print("\nConfiguration 3: 64 GPUs, TP=8, EP=8 (large MoE model)")
config3 = HybridParallelismConfig(total_gpus=64, tp_size=8, ep_size=8, dp_size=1)
config3.print_config()

In [ ]:
# Communication volume estimation
print("=== Communication Volume Estimation ===")
print("Model: DeepSeek-V3-like (hidden=7168, 61 layers, 160 experts)\n")

configs = [
    ("TP=8, EP=1", HybridParallelismConfig(8, tp_size=8)),
    ("TP=8, EP=8", HybridParallelismConfig(64, tp_size=8, ep_size=8)),
    ("TP=4, EP=8, DP=2", HybridParallelismConfig(64, tp_size=4, ep_size=8, dp_size=2)),
]

for name, config in configs:
    comm = config.estimate_communication(
        batch_size=32, seq_len=2048, hidden_dim=7168, n_layers=61, n_experts=160
    )
    print(f"{name:25s}: TP comm = {comm['tp_comm_GB']:.1f} GB, "
          f"EP comm = {comm['ep_comm_GB']:.1f} GB, "
          f"Total = {comm['total_comm_GB']:.1f} GB")

## 5. Source Code Walkthrough: SGLang Parallelism

### 5.1 Key Files

| File | Purpose |
|------|--------|
| `sglang/srt/managers/data_parallel_controller.py` | DP controller, request routing |
| `sglang/srt/model_executor/model_runner.py` | Model execution with TP |
| `sglang/srt/layers/moe/` | MoE layer implementations |
| `sglang/srt/distributed/` | Communication primitives |

### 5.2 Launch Flow for DP+TP

```python
# Simplified launch flow

def launch_server(server_args):
    if server_args.dp_size > 1:
        # Launch DataParallelController as the main process
        controller = DataParallelController(server_args)
        
        # Controller spawns DP worker groups
        for dp_rank in range(server_args.dp_size):
            # Each DP worker is a full scheduler + model_runner
            # If tp_size > 1, each DP worker spawns TP sub-processes
            gpu_ids = get_gpu_ids(dp_rank, server_args.tp_size)
            worker = launch_scheduler(
                server_args,
                dp_rank=dp_rank,
                gpu_ids=gpu_ids
            )
            controller.register_worker(worker)
        
        # Controller runs the main event loop
        controller.run()  # Routes requests to workers
    else:
        # Single scheduler (may still use TP)
        scheduler = Scheduler(server_args)
        scheduler.run()
```

### 5.3 Request Flow in DP

```
1. HTTP Server receives request
2. Tokenizer Manager tokenizes input
3. DataParallelController.dispatch_request()
   - Evaluate cache state of each worker
   - Select worker with best prefix match
4. Selected Worker's Scheduler receives request
5. Worker's Scheduler adds to waiting queue
6. Worker's Model Runner processes the batch
7. Results flow back through the controller
```

In [ ]:
# SGLang DP launch configuration example

def print_launch_config(model_name, tp, dp, ep=1, gpus_available=None):
    """Print the equivalent SGLang launch command and configuration."""
    total_gpus = tp * dp * ep
    if gpus_available and total_gpus > gpus_available:
        print(f"  ERROR: Need {total_gpus} GPUs but only {gpus_available} available")
        return
    
    print(f"\n--- {model_name} ---")
    print(f"Configuration: TP={tp}, DP={dp}, EP={ep}")
    print(f"Total GPUs needed: {total_gpus}")
    print(f"\nLaunch command:")
    cmd = f"python -m sglang.launch_server --model {model_name}"
    cmd += f" --tp {tp}"
    if dp > 1:
        cmd += f" --dp {dp}"
    if ep > 1:
        cmd += f" --ep {ep}"
    print(f"  {cmd}")
    
    print(f"\nGPU Layout:")
    gpu_id = 0
    for d in range(dp):
        for e in range(ep):
            gpus = list(range(gpu_id, gpu_id + tp))
            role = f"DP={d}"
            if ep > 1:
                role += f", EP={e}"
            print(f"  {role}: GPUs {gpus}")
            gpu_id += tp

print("=== SGLang Multi-GPU Serving Configurations ===")

# Llama-3-8B: fits on 1 GPU, scale with DP
print_launch_config("meta-llama/Llama-3-8B-Instruct", tp=1, dp=4)

# Llama-3-70B: needs TP, optionally add DP
print_launch_config("meta-llama/Llama-3-70B-Instruct", tp=4, dp=2)

# Mixtral-8x7B: TP + EP for MoE
print_launch_config("mistralai/Mixtral-8x7B-Instruct-v0.1", tp=2, dp=1, ep=4)

# DeepSeek-V3: large MoE needing all parallelism types
print_launch_config("deepseek-ai/DeepSeek-V3", tp=8, dp=1, ep=8)

## 6. Performance Scaling Analysis

### 6.1 Theoretical Scaling

**Data Parallelism scaling:**
- Ideal: linear throughput increase (2 GPUs = 2x throughput)
- Reality: slightly sub-linear due to load imbalance and routing overhead

**Tensor Parallelism scaling:**
- Ideal: linear latency decrease (2 GPUs = 0.5x latency)
- Reality: diminishing returns due to communication overhead

**The key equation for TP:**
```
Latency(TP=N) = Compute(1) / N + Communication(N)
```
Communication grows with N, so at some point adding more GPUs for TP hurts.

In [ ]:
# DP vs TP scaling model

def dp_scaling(n_gpus, overhead_per_gpu=0.02):
    """Model DP throughput scaling."""
    # Each GPU adds near-linear throughput, minus routing overhead
    efficiency = 1.0 - overhead_per_gpu * (n_gpus - 1)
    return n_gpus * max(efficiency, 0.5)

def tp_scaling(n_gpus, compute_time=10.0, comm_per_gpu=0.3):
    """Model TP latency scaling."""
    # Compute splits linearly, communication grows
    compute = compute_time / n_gpus
    communication = comm_per_gpu * (n_gpus - 1)  # All-reduce
    total_time = compute + communication
    return compute_time / total_time  # Speedup

gpu_counts = [1, 2, 4, 8, 16]

print("=== Scaling Analysis ===")
print(f"{'GPUs':>5s} | {'DP Throughput':>15s} | {'TP Speedup':>15s} | {'DP Efficiency':>15s} | {'TP Efficiency':>15s}")
print("-" * 75)

for n in gpu_counts:
    dp_tput = dp_scaling(n)
    tp_speed = tp_scaling(n)
    dp_eff = dp_tput / n
    tp_eff = tp_speed / n
    print(f"{n:5d} | {dp_tput:14.2f}x | {tp_speed:14.2f}x | {dp_eff:14.1%} | {tp_eff:14.1%}")

print("\nKey insight: DP scales throughput almost linearly.")
print("TP reduces latency but with diminishing returns beyond 8 GPUs.")

In [ ]:
# Visualize scaling comparison
if HAS_PLT:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    gpus = list(range(1, 17))
    dp_throughputs = [dp_scaling(n) for n in gpus]
    tp_speedups = [tp_scaling(n) for n in gpus]
    ideal_scaling = gpus  # Linear scaling
    
    ax1.plot(gpus, dp_throughputs, 'b-o', label='DP Throughput', linewidth=2)
    ax1.plot(gpus, ideal_scaling, 'k--', label='Ideal (linear)', alpha=0.5)
    ax1.set_xlabel('Number of GPUs')
    ax1.set_ylabel('Throughput Multiplier')
    ax1.set_title('Data Parallelism Scaling')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(gpus, tp_speedups, 'r-o', label='TP Latency Speedup', linewidth=2)
    ax2.plot(gpus, ideal_scaling, 'k--', label='Ideal (linear)', alpha=0.5)
    ax2.set_xlabel('Number of GPUs')
    ax2.set_ylabel('Speedup')
    ax2.set_title('Tensor Parallelism Scaling')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Summary

### Key Takeaways

1. **Data Parallelism** replicates the model across GPUs for throughput scaling. SGLang's `DataParallelController` manages routing with cache-aware dispatch.

2. **Expert Parallelism** distributes MoE experts across GPUs, using all-to-all communication for token dispatch and result combination.

3. **Load balancing** is critical for EP -- imbalanced expert routing leads to GPU underutilization. Token dropping and expert choice routing help.

4. **Hybrid parallelism** (TP+DP+EP) is necessary for large MoE models on many GPUs. The communication hierarchy matters: NVLink for TP, InfiniBand for EP.

5. **DP scales throughput** almost linearly. **TP reduces latency** but with diminishing returns beyond ~8 GPUs due to communication overhead.

6. **Cache-aware routing** in DP is essential to preserve the benefits of RadixAttention prefix caching.

## Exercises

### Exercise 1: Profile DP vs TP Scaling

Given a model with:
- 32 attention heads, hidden_dim=4096, 40 layers
- Prefill time (1 GPU): 50ms for 1024 tokens
- All-reduce latency: 0.1ms per operation

Calculate and compare:
1. Throughput with TP=8 (1 replica)
2. Throughput with TP=2, DP=4 (4 replicas of 2-GPU TP)
3. Throughput with TP=1, DP=8 (8 independent replicas)
4. Which configuration is best for latency-optimized serving? Throughput-optimized?

### Exercise 2: Implement Expert-Choice Routing

Implement a full expert-choice router that:
1. Each expert selects its top-C tokens (C = capacity)
2. Handles the case where a token is not selected by any expert
3. Computes the quality loss from routing changes

### Exercise 3: Cache-Aware DP Routing

Implement a sophisticated DP router that:
1. Maintains a Bloom filter summary of each worker's cache
2. Routes requests based on approximate prefix match probability
3. Falls back to least-loaded routing when no cache match exists
4. Measure improvement over round-robin

In [ ]:
# Exercise 1 Starter Code

def calculate_throughput(
    tp_size, dp_size,
    base_prefill_ms=50.0,  # Single GPU prefill time
    base_decode_ms=5.0,    # Single GPU decode step time
    allreduce_ms=0.1,      # All-reduce latency per operation
    n_layers=40,
    n_output_tokens=100,
):
    """
    TODO: Calculate throughput for the given configuration.
    
    Hints:
    - TP reduces compute per GPU but adds all-reduce communication
    - DP multiplies throughput (independent workers)
    - Each layer requires 2 all-reduce operations for TP
    """
    # YOUR CODE HERE
    
    # Compute time with TP
    prefill_compute = base_prefill_ms / tp_size
    prefill_comm = 2 * n_layers * allreduce_ms * (tp_size - 1) if tp_size > 1 else 0
    prefill_time = prefill_compute + prefill_comm
    
    decode_compute = base_decode_ms / tp_size
    decode_comm = 2 * n_layers * allreduce_ms * (tp_size - 1) if tp_size > 1 else 0
    decode_time = decode_compute + decode_comm
    
    # Total time per request
    total_time = prefill_time + n_output_tokens * decode_time
    
    # Throughput with DP
    requests_per_second_per_worker = 1000.0 / total_time
    total_throughput = requests_per_second_per_worker * dp_size
    
    return {
        "prefill_ms": prefill_time,
        "decode_ms": decode_time,
        "total_ms": total_time,
        "throughput_rps": total_throughput,
        "latency_ms": total_time  # Per-request latency
    }

# Compare configurations
configs = [
    ("TP=8, DP=1", 8, 1),
    ("TP=4, DP=2", 4, 2),
    ("TP=2, DP=4", 2, 4),
    ("TP=1, DP=8", 1, 8),
]

print(f"{'Config':<15s} | {'Prefill(ms)':>12s} | {'Decode(ms)':>11s} | {'Latency(ms)':>12s} | {'Throughput(rps)':>15s}")
print("-" * 75)
for name, tp, dp in configs:
    result = calculate_throughput(tp, dp)
    print(f"{name:<15s} | {result['prefill_ms']:11.1f} | {result['decode_ms']:10.1f} | {result['total_ms']:11.1f} | {result['throughput_rps']:14.2f}")

In [ ]:
# Exercise 3 Starter Code: Bloom Filter for Cache-Aware Routing

import hashlib

class BloomFilter:
    """Simple Bloom filter for approximate prefix matching."""
    
    def __init__(self, size=1024, n_hashes=3):
        self.size = size
        self.n_hashes = n_hashes
        self.bits = [False] * size
    
    def _hashes(self, key):
        hashes = []
        for i in range(self.n_hashes):
            h = hash((key, i)) % self.size
            hashes.append(h)
        return hashes
    
    def add(self, key):
        for h in self._hashes(key):
            self.bits[h] = True
    
    def might_contain(self, key):
        return all(self.bits[h] for h in self._hashes(key))

class BloomFilterDPRouter:
    """
    TODO: Implement a DP router using Bloom filters.
    
    Each worker maintains a Bloom filter of its cached prefixes.
    The router checks filters to find the best worker.
    """
    
    def __init__(self, n_workers):
        self.n_workers = n_workers
        self.filters = [BloomFilter() for _ in range(n_workers)]
        self.worker_loads = [0] * n_workers
    
    def route(self, prefix_key):
        # YOUR CODE HERE
        # 1. Check each worker's Bloom filter
        # 2. If match found, route to that worker
        # 3. If no match, route to least-loaded worker
        # 4. Update the chosen worker's Bloom filter
        pass

print("Exercise starter code loaded.")
print("Implement the BloomFilterDPRouter.route() method!")